# Multinomial Naive Bayes - Day 3 (From Scratch)

## What is Multinomial Naive Bayes?

Multinomial Naive Bayes is a variation of the Naive Bayes algorithm designed for discrete data. It is commonly used in text classification, where features represent word counts or frequencies.

**Key Characteristics:**
- Works with word frequencies by modeling how often words appear in a document
- Assumes a multinomial distribution for features like words
- Commonly used in spam detection, document classification, and sentiment analysis

---

## Working of Multinomial Naive Bayes

Multinomial Naive Bayes classifies text using word frequencies. Naive Bayes assumes words are independent, while Multinomial refers to counting how often words appear in a document. The model learns from training data by analyzing how often words occur in each class.

---

## Multinomial Distribution Formula

$$P(X|C) = \frac{n!}{n_1! n_2! \ldots n_m!} p_1^{n_1} p_2^{n_2} \ldots p_m^{n_m}$$

Where:
- $n$ is the total number of trials
- $n_i$ is the count of occurrences for outcome i
- $p_i$ is the probability of outcome i

---

## Laplace Smoothing

To avoid zero probability, we apply Laplace smoothing:

$$\theta_{c,i} = \frac{count(w_i, c) + 1}{N + V}$$

Where:
- $count(w_i, c)$ is the number of times word $w_i$ appears in documents of class $c$
- $N$ is the total number of words in documents of class $c$
- $V$ is the vocabulary size

---

## One-Line Summary

**Multinomial Naive Bayes uses word frequencies and Laplace smoothing to classify text documents.**

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

print("="*50)
print("MULTINOMIAL NAIVE BAYES DAY 3 - FROM SCRATCH")
print("="*50)

## Multinomial Naive Bayes Implementation from Scratch

In [ ]:
class MultinomialNaiveBayes:
    """
    Multinomial Naive Bayes Classifier from Scratch
    """
    
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # Laplace smoothing parameter
        self.classes = None
        self.class_priors = {}
        self.feature_probs = {}
        self.vocabulary_size = 0
    
    def fit(self, X, y):
        """
        Train the Multinomial Naive Bayes classifier
        """
        self.classes = np.unique(y)
        n_samples = len(y)
        
        # Calculate class priors
        for cls in self.classes:
            self.class_priors[cls] = np.sum(y == cls) / n_samples
        
        # Calculate feature probabilities for each class
        for cls in self.classes:
            X_cls = X[y == cls]
            
            # Total word count in class
            total_words = np.sum(X_cls, axis=0).sum()
            
            # Sum of word counts per feature
            feature_counts = np.sum(X_cls, axis=0)
            self.vocabulary_size = X.shape[1]
            
            # Laplace smoothing
            smoothed_probs = (feature_counts + self.alpha) / (total_words + self.alpha * self.vocabulary_size)
            self.feature_probs[cls] = smoothed_probs
    
    def predict_single(self, x):
        """
        Predict a single sample
        """
        posteriors = {}
        
        for cls in self.classes:
            # Start with prior
            posterior = np.log(self.class_priors[cls])
            
            # Multiply by feature probabilities
            for i in range(len(x)):
                if x[i] > 0:
                    posterior += x[i] * np.log(self.feature_probs[cls][i])
            
            posteriors[cls] = posterior
        
        return max(posteriors, key=posteriors.get)
    
    def predict(self, X):
        return np.array([self.predict_single(x) for x in X])
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

## Example: Spam Detection Dataset

In [ ]:
# Create dataset
data = {
    'text': [
        'Free money now',
        'Call now to claim your prize',
        'Meet me at the park',
        'Lets catch up later',
        'Win a new car today',
        'Lunch plans',
        'Congratulations You won a lottery',
        'Can you send me the report',
        'Exclusive offer for you',
        'Are you coming to the meeting'
    ],
    'label': ['spam', 'spam', 'not spam', 'not spam', 'spam', 
              'not spam', 'spam', 'not spam', 'spam', 'not spam']
}

df = pd.DataFrame(data)
print("Dataset:")
print(df)

In [ ]:
# Build vocabulary
def build_vocabulary(texts):
    vocabulary = set()
    for text in texts:
        words = text.lower().split()
        vocabulary.update(words)
    return sorted(list(vocabulary))

vocabulary = build_vocabulary(df['text'])
print(f"Vocabulary size: {len(vocabulary)}")
print("Vocabulary:", vocabulary)

In [ ]:
# Convert text to word count vectors
def text_to_vector(text, vocabulary):
    words = text.lower().split()
    word_count = Counter(words)
    vector = [word_count.get(word, 0) for word in vocabulary]
    return np.array(vector)

X = np.array([text_to_vector(text, vocabulary) for text in df['text']])
y = np.array([1 if label == 'spam' else 0 for label in df['label']])

print("Feature matrix shape:", X.shape)
print("\nWord count vectors (first 2 documents):")
for i in range(2):
    print(f"Document {i+1}: {X[i]}")

In [ ]:
# Split data
split = int(0.7 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

In [ ]:
# Train Multinomial Naive Bayes
mnb = MultinomialNaiveBayes(alpha=1.0)
mnb.fit(X_train, y_train)

print("\nClass Priors:")
for cls in mnb.classes:
    class_name = "Spam" if cls == 1 else "Not Spam"
    print(f"P({class_name}) = {mnb.class_priors[cls]:.4f}")

In [ ]:
# Predictions
y_pred = mnb.predict(X_test)
accuracy = mnb.score(X_test, y_test)

print("\nPredictions:")
for i in range(len(X_test)):
    actual = "Spam" if y_test[i] == 1 else "Not Spam"
    pred = "Spam" if y_pred[i] == 1 else "Not Spam"
    match = "✓" if y_pred[i] == y_test[i] else "✗"
    print(f"Actual: {actual:<8} Predicted: {pred:<8} {match}")

print(f"\nAccuracy: {accuracy*100:.2f}%")

In [ ]:
# Test custom message
custom_message = "Congratulations you won a free vacation"
custom_vector = text_to_vector(custom_message, vocabulary)
prediction = mnb.predict_single(custom_vector)

print("\nCustom Message Prediction:")
print(f"Message: '{custom_message}'")
print(f"Prediction: {'Spam' if prediction == 1 else 'Not Spam'}")

In [ ]:
# Day 3 Completed
print("\n" + "="*50)
print("MULTINOMIAL NAIVE BAYES DAY 3 COMPLETED")
print("="*50)
print("Topics covered:")
print("- Multinomial Naive Bayes definition")
print("- Multinomial distribution formula")
print("- Laplace smoothing")
print("- Spam detection example")
print("- Vocabulary building")
print("- Text to vector conversion")
print("- Custom message prediction")
print("="*50)"
print("\nNext: Day 4 - Bernoulli Naive Bayes")